# Inference with Denoising Diffusion Implicit Model (DDIM) <br> Hugging Face `diffusers` library

Source:<P>

https://github.com/huggingface/diffusers/tree/main/examples

Adapted:<P>

Antonio Esteves @ UMinho, April 2025<P>

---

Diffusion models proved themselves very effective in artificial synthesis, even beating GANs for images. Because of that, they gained traction in the machine learning community and play an important role for systems like [DALL-E 2](https://openai.com/dall-e-2/) or [Imagen](https://imagen.research.google/) to generate photorealistic images when prompted on text.

While the most prolific successes of diffusion models have been in the computer vision community, these models have also achieved remarkable results in other domains, such as:
- [video generation](https://video-diffusion.github.io/),
- [audio synthesis](https://diffwave-demo.github.io/),
- [reinforcement learning](https://diffusion-planning.github.io/),
- and more.

However, most of the recent research on diffusion models, *e.g.* DALL-E 2 and Imagen, is unfortunately not accessible to the broader machine learning community and typically remains behind closed doors.

Here comes the `diffusers` library with the goals to:

1. gather recent diffusion models from independent repositories in a single and **long-term maintained** project that is built by and for the **community**,
2. reproduce high impact machine learning systems such as DALLE and Imagen in a manner that is accessible for the public, and
3. create an easy to use API that enables one to train their own models or re-use checkpoints from other repositories for inference.

This notebook goes through the most important features of `diffusers`.

## Summary

The notebook illustrates the core API of `diffusers`, which is divided into three components:
1. **Pipelines**: high-level classes designed to rapidly generate samples from popular trained diffusion models in a user-friendly fashion.
2. **Models**: popular architectures for training new diffusion models, *e.g.* [UNet](https://arxiv.org/abs/1505.04597).
3. **Schedulers**: various techniques for generating images from noise during *inference* as well as to generate noisy images for *training*.

**Note**: This notebook focus only on **inference**.

## Install `diffusers`

In [ ]:
# !pip install diffusers

## Overview

One goal of the diffusers library is to make diffusion models accessible to a wide range of deep learning practitioners.

As a quick recap, diffusion models are trained to *denoise* random gaussian noise step by step, to get to a sample of interest, such as an *image*.

The underlying model, often a neural network, is trained to predict a way to slightly denoise the image in each step. After certain number of steps, a sample is obtained.

The process is illustrated by the following figure.
![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusion-process.png)

The architecture of the neural network, referred to as **model**, commonly follows the UNet architecture as proposed in [this paper](https://arxiv.org/abs/1505.04597) and improved upon in the Pixel++ paper.

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/unet-model.png)

Some of the characteristics of the architecture are:
* this model outputs images of the same size as the input
* the model passes the input image through several ResNet blocks that halve the feature maps size by 2
* then the feature maps go through the same number of blocks that upsample them.
* skip connections link features on the downsample path to corresponding layers in the upsample path.

The diffusion process consists in taking random noise of the size of the desired output and pass it through the model several times. The process ends after a given number of steps, and the output image should represent a sample according to the training data distribution of the model, for instance an image of a butterfly.

During training we show many samples of a given distribution, such as images of butterfly. After training, the model will be able to process random noise to generate similar butterfly images.

In a diffusion model such as DDPM, the model is not trained to directly predict a slightly less noisy image, but rather to predict the noise that should be removed from the noisy image or, similarly, the gradient between the two time steps (like the diffusion model called "Score VE").

To do the denoising process, a specific noise scheduling algorithm is thus necessary and "wrap" the model to define how many diffusion steps are needed for inference as well as how to compute a less noisy image from the model's output. Here is where the different **schedulers** of the diffusers library come into play.

Finally, a **pipeline** groups together a **model** and a **scheduler** and make it easy for an end-user to run a full denoising loop process. W will start with the pipelines and dive deeper into its implementation before taking a closer look at models and schedulers.

## `diffusers` API

### Pipelines

Let us begin by importing a pipeline. We will use the `google/ddpm-celebahq-256` model, built in collaboration by Google and U.C.Berkeley. It is a model following the [Denoising Diffusion Probabilistic Models (DDPM) algorithm](https://arxiv.org/abs/2006.11239) trained on a dataset of celebrities images.

We can import the `DDPMPipeline`, which will allow us to do inference with a couple of lines of code.

In [ ]:
from diffusers import DDPMPipeline

The `from_pretrained()` method allows downloading the model and its configuration from [the Hugging Face Hub](https://huggingface.co/google/ddpm-celebahq-256), a repository of over 60,000 models shared by the community.


In [ ]:
repo_id1   = "google/ddpm-celebahq-256"
image_pipe = DDPMPipeline.from_pretrained(repo_id1)
image_pipe.to("cuda")

To generate an image, we simply run the pipeline, which uses a random initial noise sample and then iterates through the diffusion process.

The pipeline returns as output a dictionary with a generated sample.

In [ ]:
images = image_pipe().images

Let us visualize the sampled image 🙂.

In [ ]:
images[0]

Now, let us try to understand what the pipeline is made of.

In [ ]:
image_pipe

We can see inside the pipeline a scheduler and a UNet model. Let us have a closer look at them and what this pipeline just did behind the scenes.

## Models

Instances of the model class are neural networks that take a noisy `sample` as well as a `timestep` as inputs to predict a less noisy output `sample`. Let us load a pre-trained model and play around with it to understand the model API!

We will load a simple unconditional image generation model of type `UNet2DModel` which was released with the [DDPM Paper](https://arxiv.org/abs/2006.11239) and take a look at another model trained on church images: [`google/ddpm-church-256`](https://huggingface.co/google/ddpm-church-256).

Similarly to what we have seen for the pipeline class, we can load the model configuration and weights with one line, using the `from_pretrained()` method.

In [ ]:
from diffusers import UNet2DModel

#repo_id2 = "google/ddpm-church-256"
model   = UNet2DModel.from_pretrained(repo_id1)

The `from_pretrained()` method caches the model weights locally, so if you execute the cell above a second time, it will go much faster. The model is a pure PyTorch `torch.nn.Module` class which you can see when printing out `model`.

In [ ]:
model

Now let us take a look at the model's configuration. The configuration can be accessed via the `config` attribute and shows all the necessary parameters to define the model architecture.

In [ ]:
model.config

As you can see, the model configuration is a frozen dictionary. This is to enforce that the configuration will only be used to define the model architecture at instantiation time and not for any attributes that can be changed during inference.

A couple of important configuration parameters are:
- `sample_size`: defines the `height` and `width` dimension of the input sample.
- `in_channels`: defines the number of input channels of the input sample.
- `down_block_types` and `up_block_types`: define the type of down- and upsampling blocks that are used to create the UNet architecture as was seen in the figure at the beginning of this notebook.
- `block_out_channels`: defines the number of output channels of the downsampling blocks, also used in reversed order for the number of input channels of the upsampling blocks.
- `layers_per_block`: defines how many ResNet blocks are present in each UNet block.

Knowing how a U-Net configuration looks like, we can quickly try to instantiate the exact same model architecture with random weights. To do so, let us pass the configuration as an unpacked dictionary to the `UNet2DModel` class.

In [ ]:
model_random = UNet2DModel(**model.config)

So, we created a randomly initialized model with the same configuration as the previously loaded from Hugging Face Hub.

If we want to save the model just created, we can use the `save_pretrained` method, which saves both the model weights as well as the model configuration in the provided folder.

In [ ]:
model_random.save_pretrained("local_unet")

Let us take a look at what files were saved in `local_unet`.

In [ ]:
!ls local_unet

`diffusion_pytorch_model.safetensors` is a binary PyTorch file that stores the model weights and `config.json` stores the model's configuration.

And we can reuse the model, by simply using the `from_pretrained()` method again, as it loads local checkpoints as well as those present on the Hub.

In [ ]:
model_random = UNet2DModel.from_pretrained("local_unet")

Let us now return our attention to the model loaded from the Hub and see how we can use it for inference. First, we need a random gaussian sample in the shape of an image (`batch_size` x `in_channels` x `sample_size` x `sample_size`). We have a `batch` dimension because a model can receive multiple random noise samples, a `channel` dimension because each random noise sample consists of multiple channels, and finally `sample_size` corresponds to the height and width.

In [ ]:
import torch

torch.manual_seed(34)

noisy_sample = torch.randn(
    1,
    model.config.in_channels,
    model.config.sample_size,
    model.config.sample_size
)
noisy_sample.shape

## Inference

Besides the noise sample(s), we can also pass the current timestep(s) through the model. The timestep is an important cue about how noisy the input image is, more noisy in the beginning of the process and less noisy at the end, so the model knows if it is closer to the start or the end of the diffusion process.

As explained in the introduction, the model predicts either the slightly less noisy image, the difference between the slightly less noisy image and the input image or even something else. It is important to carefully read through the [model card](https://huggingface.co/google/ddpm-church-256) to know what the model has been trained on. In this case, the model predicts the noise difference between the slightly less noisy image and the input image.

In [ ]:
with torch.no_grad():
    noise_residual = model(sample=noisy_sample, timestep=2).sample

The predicted `noise_residual` has the exact same shape as the input and we use it to compute a slightly less noisy image. Let us confirm the output shapes.

In [ ]:
noise_residual.shape

Now to summarize, **models**, such as `UNet2DModel` are parameterized neural networks trained to predict a slightly less noisy image or the noise residual. They are defined by their `.config` and can be loaded from the Hub as well as saved and loaded locally. The next step is learning how to combine this **model** with the correct **scheduler** to be able to actually generate images.

## Schedulers

**Schedulers** are algorithms wrapped into a Python class. They define the noise schedule which is used to add noise to the model during training, and also define the algorithm to compute the slightly less noisy sample given the model output (here `noise_residual`). This notebook focuses only on how to use *scheduler* classes for inference.

It is important to stress here that while *models* have trainable weights, *schedulers* are usually *parameter-free*, in the sense they have no trainable weights, and simply define the algorithm to compute the slightly less noisy sample. Schedulers thus do not inherit from `torch.nn.Module`, but like models they are instantiated by a configuration.

To download a scheduler configuration from the Hub, we can make use of the `from_config()` method to load a configuration and instantiate a scheduler.

Let us use `DDPMScheduler`, the denoising algorithm proposed in the [DDPM Paper](https://arxiv.org/abs/2006.11239).

In [ ]:
from diffusers import DDPMScheduler

scheduler = DDPMScheduler.from_config(repo_id1)

Let us also take a look at the configuration of the scheduler.

In [ ]:
scheduler.config

Different schedulers are usually defined by different parameters. To better understand what the parameters are used for exactly, we can take a look at the respective scheduler files under `src/diffusers/schedulers/`, such as the [`src/diffusers/schedulers/scheduling_ddpm.py`](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_ddpm.py) file. Here are the most important ones:
- `num_train_timesteps` defines the length of the denoising process, e.g. how many timesteps are need to process random gaussian noise to a data sample.
- `beta_schedule` define the type of noise schedule that shall be used for inference and training
- `beta_start` and `beta_end` define the smallest noise value and highest noise value of the schedule.

Like the *models*, *schedulers* can be saved and loaded with `save_config()` and `from_config()`.



In [ ]:
scheduler.save_config("local_ddpm_scheduler")
new_scheduler = DDPMScheduler.from_config("local_ddpm_scheduler")

All schedulers provide one or multiple `step()` methods that can be used to compute the slightly less noisy image. The `step()` method may vary from one scheduler to another, but normally expects at least the model output, the `timestep` and the current `noisy_sample`.

Let us use the model output from the previous section.

In [ ]:
less_noisy_sample = scheduler.step(
    model_output = noise_residual,
    timestep     = 2,
    sample       = noisy_sample,
).prev_sample
less_noisy_sample.shape

So, the computed sample has the exact same shape as the model input, meaning that we are ready to pass it to the model again in a next step.

We can now put all together and actually define the denoising loop. This loop prints out the (less and less) noisy samples along the way for better visualization in the denoising loop. Let us define a display function that takes care of post-processing the denoised image, convert it to a `PIL.Image` and displays it.

In [ ]:
import PIL.Image
import numpy     as np

def display_sample(sample, i):
    image_processed = sample.cpu().permute(0, 2, 3, 1)
    image_processed = (image_processed + 1.0) * 127.5
    image_processed = image_processed.numpy().astype(np.uint8)

    image_pil = PIL.Image.fromarray(image_processed[0])
    display(f"Image at step {i}")
    display(image_pil)

Before defining the loop, we place the input and model in the GPU to speed up the denoising process.

In [ ]:
model.to("cuda")
noisy_sample = noisy_sample.to("cuda")

The denoising loop is straight-forward for DDPM.

1. Predict the residual of the less noisy sample with the model.
2. Compute the less noisy sample with the scheduler.

Additionally, at every 50 steps it will display the progress.

It is important to note here that we iterate over `scheduler.timesteps`, which is a tensor defining the sequence of timesteps over which to iterate during the denoising process. Usually, the denoising process goes in decreasing order of timesteps, so from the total number of timesteps (here 1000) downto 0.

In [ ]:
import tqdm

sample = noisy_sample

for i, t in enumerate(tqdm.tqdm(scheduler.timesteps)):
  # 1. predict noise residual
  with torch.no_grad():
      residual = model(sample, t).sample

  # 2. compute less noisy image and set x_t -> x_t-1
  sample = scheduler.step(residual, t, sample).prev_sample

  # 3. optionally display the image
  if (i + 1) % 50 == 0:
      display_sample(sample, i + 1)

One can see that it takes some time to see a somewhat meaningful shape, only after approximately 800 steps.

While the quality of the image is actually quite good, we might want to speed up the image generation.

To do so, we can replace the DDPM scheduler with the [DDIM](https://arxiv.org/abs/2010.02502) scheduler which keeps high generation quality at significantly sped-up generation time.

**Exchanging schedulers**: in the `diffusers` library, we can use different schedulers with the same model. In this case, DDIM can replace DDPM.

The DDPM and DDIM schedulers more or less share the same configuration, so we can load a DDIM scheduler from a DDPM scheduler.

In [ ]:
from diffusers import DDIMScheduler

scheduler = DDIMScheduler.from_config(repo_id1)

The DDIM scheduler allows us to define how many denoising steps should be run at inference via the `set_timesteps` method. The DDPM scheduler runs by default 1000 denoising steps. Let us significantly reduce this number to just 50 inference steps for DDIM. And we can run the same loop as before, but now it makes use of the much faster DDIM scheduler.

And we can run the same loop as before, but now it makes use of the much faster DDIM scheduler.

In [ ]:
import tqdm

# We will try different numbers of timesteps in DDIM 
list_timesteps = [50, 100, 200, 500, 1000]

for ts in list_timesteps:

    scheduler.set_timesteps(num_inference_steps = ts)
    
    sample = noisy_sample
    
    for i, t in enumerate(tqdm.tqdm(scheduler.timesteps)):
      # 1. predict noise residual
      with torch.no_grad():
          residual = model(sample, t).sample
    
      # 2. compute previous image and set x_t -> x_t-1
      sample = scheduler.step(residual, t, sample).prev_sample
    
      # 3. optionally display the image
      if (i + 1) % 5 == 0:
          display_sample(sample, i + 1)

We observe that the image generation is indeed much faster, a mere two seconds, but the image quality has decreased.

Cool, we now have the basic understanding of schedulers. The important things to remember are:
1. schedulers are *parameter-free* (no trainable weights).
2. schedulers define the algorithm computing the slightly less noisy sample during inference.

There are many schedulers already in `diffusers`. It is important to read the model card to understand which model checkpoints can be used with which schedulers.
The available schedulers can be accessed [here](https://github.com/huggingface/diffusers/tree/main/src/diffusers/schedulers).

Also note that `diffusers` tries to keep *models* and *schedulers* as independent from each other as possible. This means a `scheduler` should never accept a `model` as an input and vice-versa. The model *predicts* the noise residual or slightly less noisy image with its trained weights, while the scheduler *computes* the previous sample given the model's output.